# 🧠 Desarrollo de Modelos - Proyecto Orion

## Objetivo
Explorar el rendimiento de los diferentes modelos:
- Fase 1: Isolation Forest
- Fase 2: Random Forest
- Fase 3: Ensemble (XGBoost + LightGBM)

Y comparar su rendimiento.

In [ ]:
# ============================================================
# IMPORTS Y CONFIGURACIÓN
# ============================================================

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import joblib

from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                              f1_score, confusion_matrix, roc_curve, auc)
from sklearn.model_selection import train_test_split

from src.utilidades.tiempo import ahora_peru

plt.style.use('seaborn-v0_8-darkgrid')

print(f"📅 Hora Perú: {ahora_peru().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ Librerías cargadas")

## 1. Cargar Datos de Entrenamiento

Cargamos los datos etiquetados (features + labels).

In [ ]:
# ============================================================
# CARGAR DATOS DE ENTRENAMIENTO
# ============================================================

train_path = Path('../datos/retroalimentacion/entrenamiento/datos_etiquetados.parquet')

if train_path.exists():
    df = pd.read_parquet(train_path)
    print(f"✅ Datos cargados: {len(df)} registros")
    print(f"📊 Columnas: {df.columns.tolist()}")
    
    if 'label' in df.columns:
        print(f"\n📊 Distribución de labels:")
        print(df['label'].value_counts())
    
    display(df.head())
else:
    print("⚠️ No hay datos de entrenamiento")
    print("💡 Ejecuta: python scripts/ejecutar_flujo_completo.py")

## 2. Métricas del Modelo Actual

Revisamos las métricas del modelo entrenado actualmente.

In [ ]:
# ============================================================
# MÉTRICAS DEL MODELO ACTUAL
# ============================================================

metrics_path = Path('../modelos/actual/metrics.json')

if metrics_path.exists():
    with open(metrics_path, 'r', encoding='utf-8') as f:
        metricas = json.load(f)
    
    print("=" * 60)
    print("🧠 MÉTRICAS DEL MODELO ACTUAL")
    print("=" * 60)
    
    # Mostrar métricas
    for key in ['version', 'fecha', 'tipo_modelo', 'n_muestras']:
        if key in metricas:
            print(f"   {key}: {metricas[key]}")
    
    print()
    for key in ['accuracy', 'precision', 'recall', 'f1']:
        if key in metricas:
            print(f"   {key}: {metricas[key]*100:.2f}%")
    
    # Visualizar
    fig, ax = plt.subplots(figsize=(10, 5))
    
    metricas_viz = {k: metricas.get(k, 0) * 100 
                    for k in ['accuracy', 'precision', 'recall', 'f1']}
    
    bars = ax.bar(metricas_viz.keys(), metricas_viz.values(),
                  color=['#4dabf7', '#51cf66', '#ffd93d', '#ff6b6b'])
    ax.set_title(f'Métricas del Modelo ({metricas.get("tipo_modelo", "N/A")})', 
                 fontsize=14, fontweight='bold')
    ax.set_ylabel('Porcentaje (%)')
    ax.set_ylim(0, 100)
    
    for bar, value in zip(bars, metricas_viz.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{value:.1f}%', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No hay métricas del modelo")

## 3. Evaluación Detallada con Datos de Entrenamiento

Evaluamos el modelo usando el conjunto de datos de entrenamiento.

In [ ]:
# ============================================================
# EVALUACIÓN DEL MODELO
# ============================================================

if train_path.exists() and metrics_path.exists():
    # Cargar modelo
    modelo_path = Path('../modelos/actual/modelo_actual.pkl')
    
    if modelo_path.exists():
        modelo_data = joblib.load(modelo_path)
        
        if isinstance(modelo_data, dict):
            modelo = modelo_data.get('modelo')
        else:
            modelo = modelo_data
        
        # Obtener features
        features_modelo = metricas.get('features_usadas', [])
        features_modelo = [f for f in features_modelo if f in df.columns]
        
        if not features_modelo:
            # Inferir features
            columnas_excluir = ['id_cliente', 'resultado', 'label', 
                                'fecha_procesamiento', 'fecha_inspeccion',
                                'tipo_irregularidad', 'descripcion', 
                                'inspector', 'cnr_estimado', 'monto_recuperar']
            features_modelo = [col for col in df.columns 
                              if col not in columnas_excluir 
                              and df[col].dtype in ['float64', 'int64']]
        
        print(f"📊 Features usadas: {features_modelo}")
        
        # Preparar datos
        X = df[features_modelo].fillna(0).values
        y = df['label'].values
        
        # Predicciones
        if hasattr(modelo, 'predecir'):
            y_prob = modelo.predecir(X)
        else:
            y_prob = modelo.predict_proba(X)[:, 1]
        
        y_pred = (y_prob > 0.5).astype(int)
        
        # Métricas
        print("\n📊 MÉTRICAS DE EVALUACIÓN")
        print("=" * 60)
        print(f"   Accuracy:  {accuracy_score(y, y_pred)*100:.2f}%")
        print(f"   Precision: {precision_score(y, y_pred, zero_division=0)*100:.2f}%")
        print(f"   Recall:    {recall_score(y, y_pred, zero_division=0)*100:.2f}%")
        print(f"   F1-Score:  {f1_score(y, y_pred, zero_division=0)*100:.2f}%")
        
        # Matriz de confusión
        cm = confusion_matrix(y, y_pred)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap='Blues')
        plt.title('Matriz de Confusión', fontsize=14, fontweight='bold')
        plt.colorbar()
        
        clases = ['Normal', 'Hurto']
        tick_marks = np.arange(len(clases))
        plt.xticks(tick_marks, clases)
        plt.yticks(tick_marks, clases)
        
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black",
                        fontsize=14)
        
        plt.ylabel('Real')
        plt.xlabel('Predicho')
        plt.tight_layout()
        plt.show()
        
        print(f"\n📊 Matriz de Confusión:")
        print(f"   Verdaderos Negativos: {cm[0,0]}")
        print(f"   Falsos Positivos:     {cm[0,1]}")
        print(f"   Falsos Negativos:     {cm[1,0]}")
        print(f"   Verdaderos Positivos: {cm[1,1]}")
    else:
        print("⚠️ No hay modelo entrenado")
else:
    print("⚠️ Faltan datos de entrenamiento o métricas")

## 4. Comparación con Datos de Muestra

Si tenemos datos de muestra, podemos probar el modelo con ellos.

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE PROBABILIDADES
# ============================================================

if train_path.exists() and modelo_path.exists():
    plt.figure(figsize=(12, 5))
    
    # Histograma de probabilidades
    plt.subplot(1, 2, 1)
    plt.hist(y_prob[y == 0], bins=30, alpha=0.6, label='Normal', color='#6bcb77')
    plt.hist(y_prob[y == 1], bins=30, alpha=0.6, label='Hurto', color='#ff6b6b')
    plt.axvline(x=0.5, color='black', linestyle='--', label='Umbral')
    plt.title('Distribución de Probabilidades')
    plt.xlabel('Probabilidad')
    plt.ylabel('Frecuencia')
    plt.legend()
    
    # Curva ROC
    plt.subplot(1, 2, 2)
    fpr, tpr, thresholds = roc_curve(y, y_prob)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, color='#4dabf7', lw=2, 
             label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos')
    plt.ylabel('Tasa de Verdaderos Positivos')
    plt.title('Curva ROC')
    plt.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 ROC-AUC: {roc_auc:.3f}")

## 5. Resumen del Análisis de Modelos

Conclusiones del análisis.

In [ ]:
# ============================================================
# RESUMEN
# ============================================================

print("=" * 60)
print("📊 RESUMEN DEL ANÁLISIS DE MODELOS")
print("=" * 60)
print(f"📅 Fecha: {ahora_peru().strftime('%Y-%m-%d %H:%M:%S')} (Perú)")
print()

if metrics_path.exists():
    print(f"📦 Versión: {metricas.get('version', 'N/A')}")
    print(f"🧠 Tipo: {metricas.get('tipo_modelo', 'N/A')}")
    print(f"📊 Muestras: {metricas.get('n_muestras', 'N/A')}")
    print()
    print(f"🎯 Accuracy:  {metricas.get('accuracy', 0)*100:.2f}%")
    print(f"📊 Precision: {metricas.get('precision', 0)*100:.2f}%")
    print(f"📈 Recall:    {metricas.get('recall', 0)*100:.2f}%")
    print(f"⚡ F1-Score:  {metricas.get('f1', 0)*100:.2f}%")

print("\n" + "=" * 60)
print("✅ Análisis de modelos completado")
print("=" * 60)